In [0]:
# =========================
# Imports
# =========================
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re

print("Starting Silver transformation...")

# =========================
# Date parsing function
# =========================
def parse_date(col):
    col_ref = F.col(col) if isinstance(col, str) else col
    return F.coalesce(
        F.try_to_date(col_ref, "yyyy-MM-dd"),
        F.try_to_date(col_ref, "M/d/yyyy"),
        F.try_to_date(col_ref, "MM-dd-yyyy"),
        F.try_to_date(col_ref, "M-d-yyyy"),
        F.try_to_date(col_ref, "yyyy/MM/dd")
    )

# =========================
# Null handling function
# =========================
def apply_null_rules(df):
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit("UNKNOWN")))
        elif dtype in ("int", "bigint", "long"):
            if col_name not in ["customer_key", "product_key", "store_key", "order_number"]:
                df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0)))
        elif dtype in ("float", "double"):
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0.0)))
        elif "decimal" in dtype:
            df = df.withColumn(col_name, F.coalesce(F.col(col_name), F.lit(0).cast(dtype)))
    return df

In [0]:
print("Processing customers...")

df_customers = spark.table("01_prod_bronze.raw.customers")

df_customers = df_customers \
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), -1)
         .otherwise(F.col("customer_key").cast(IntegerType()))
    ) \
    .withColumn("birthday", parse_date("birthday")) \
    .withColumn("gender",
        F.when(F.col("gender").isin("Male","Female"), F.col("gender"))
         .otherwise("unknown")
    ) \
    .withColumn("country", F.initcap(F.trim(F.col("country")))) \
    .withColumn("continent", F.initcap(F.trim(F.col("continent")))) \
    .withColumn("state", F.initcap(F.trim(F.col("state")))) \
    .withColumn("state_code", F.trim(F.col("state_code"))) \
    .withColumn("state_code",
        F.when(F.col("state_code").isNull(),
               F.upper(F.substring(F.col("state"), 1, 3)))
         .when(F.length(F.col("state_code")) <= 3,
               F.upper(F.col("state_code")))
         .otherwise(F.upper(F.substring(F.col("state_code"), 1, 3)))
    )

df_customers = apply_null_rules(df_customers)

In [0]:
print("Processing products...")

df_products = spark.table("01_prod_bronze.raw.products")

df_products = df_products \
    .withColumn("unit_price_usd",
        F.regexp_replace(F.trim(F.col("unit_price_usd")), "[$,]", "")
    ) \
    .withColumn("unit_cost_usd",
        F.regexp_replace(F.trim(F.col("unit_cost_usd")), "[$,]", "")
    ) \
    .withColumn("unit_price_usd",
        F.when(F.col("unit_price_usd") == "", None)
         .otherwise(F.col("unit_price_usd"))
         .cast(DecimalType(18,2))
    ) \
    .withColumn("unit_cost_usd",
        F.when(F.col("unit_cost_usd") == "", None)
         .otherwise(F.col("unit_cost_usd"))
         .cast(DecimalType(18,2))
    ) \
    .withColumn("category", F.initcap(F.trim(F.col("category")))) \
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory")))) \
    .dropDuplicates(["product_key"])

df_products = apply_null_rules(df_products)

In [0]:
print("Processing stores...")

df_stores = spark.table("01_prod_bronze.raw.stores")

df_stores = df_stores \
    .withColumn("store_key", F.col("store_key").cast(IntegerType())) \
    .filter(F.col("store_key").isNotNull()) \
    .withColumn("open_date", parse_date("open_date")) \
    .withColumn("country", F.initcap(F.trim(F.col("country")))) \
    .withColumn("state", F.initcap(F.trim(F.col("state")))) \
    .dropDuplicates(["store_key"])

df_stores = apply_null_rules(df_stores)

In [0]:
print("Processing exchange rates...")

df_exchange_rates = spark.table("01_prod_bronze.raw.exchange_rates")

df_exchange_rates = df_exchange_rates \
    .withColumn("date", parse_date("date")) \
    .withColumn("currency", F.upper(F.trim(F.col("currency")))) \
    .withColumn("exchange_rate",
        F.regexp_replace(F.trim(F.col("exchange")), ",", ".")
    ) \
    .withColumn("exchange_rate",
        F.when(F.col("exchange_rate") == "", None)
         .otherwise(F.col("exchange_rate"))
         .cast(DecimalType(18,6))
    ) \
    .drop("exchange") \
    .dropDuplicates(["date", "currency"])

df_exchange_rates = apply_null_rules(df_exchange_rates)

df_exchange_rates = df_exchange_rates.withColumn(
    "exchange_rate",
    F.when(F.col("exchange_rate") == 0, None)
     .otherwise(F.col("exchange_rate"))
)

In [0]:
print("Processing sales...")

df_sales = spark.table("01_prod_bronze.raw.sales")

df_sales = df_sales \
    .withColumn("order_number",
        F.when(F.col("order_number").isNull(), -1)
         .otherwise(F.col("order_number"))
    ) \
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), -1)
         .otherwise(F.col("customer_key"))
    ) \
    .withColumn("store_key",
        F.when(F.col("store_key").isNull(), -1)
         .otherwise(F.col("store_key"))
    ) \
    .withColumn("product_key",
        F.when(F.col("product_key").isNull(), -1)
         .otherwise(F.col("product_key"))
    ) \
    .withColumn("order_date", parse_date("order_date")) \
    .withColumn("delivery_date", parse_date("delivery_date")) \
    .withColumn("currency_code", F.upper(F.trim(F.col("currency_code")))) \
    .withColumn("quantity",
        F.when(F.col("quantity") < 0, None).otherwise(F.col("quantity"))
    ) \
    .withColumn("delivery_date",
        F.when(F.col("delivery_date").isNull(), F.col("order_date"))
         .when(F.col("delivery_date") < F.col("order_date"), F.col("order_date"))
         .otherwise(F.col("delivery_date"))
    )

df_sales = df_sales.fillna({"currency_code": "UNKNOWN"})
df_sales = apply_null_rules(df_sales)

In [0]:
print("Writing all tables to Silver layer...")

tables = {
    "customers": df_customers,
    "products": df_products,
    "stores": df_stores,
    "sales": df_sales,
    "exchange_rates": df_exchange_rates
}

for name, df in tables.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"02_prod_silver.transformed.{name}")
    
    print(f"Table written: {name}")

print("Silver transformation completed successfully")